# Instacart Data Pipeline
## Stage 4: Analytics — Business Questions

**Owner:** Maeve  
**Business-data source:** `workspace.instacart_gold`  
**Output schema:** `workspace.instacart_analytics`

### What this notebook does

Answers the three assignment questions and the fourth team question using
five numbered tasks (21–25). It builds six business-question output tables
plus one KPI table, then runs a consolidated sanity check across all
**seven** outputs.

Business data comes from Gold, never Bronze or Silver. Within Analytics,
the daily basket summary reuses the day/hour table, and validation reads
the generated output tables.

### Business questions covered

1. Which products and departments are purchased most frequently?
2. How does customer purchasing behavior change by day of week and hour of day?
3. Which products have the highest reorder behavior?
4. Which products are most often bought together?

### Prerequisites

The shared Setup must already have created `workspace.instacart_analytics`.
The following Gold tables must exist and have valid source keys and
relationships:

- `gold_fact_order_product`: product-line events with `order_id`,
  `product_id`, `add_to_cart_order`, and the boolean `reordered` flag.
- `gold_dim_product`: product names, aisle names, and department context.
- `gold_dim_order`: order identifiers and day/hour context.

The notebook does not create the Analytics schema. Every task (21-25)
explicitly selects the target catalog and schema, so tasks can also run
independently, not only top-to-bottom in the same session.

### Build Order

| Task | Name | Output tables | Depends on |
|---|---|---|---|
| 21 | Product and Department Frequency | `analytics_top_departments`, `analytics_top_products` | Gold fact and product dimension |
| 22 | Customer Behavior by Day and Hour | `analytics_day_hour_patterns`, `analytics_basket_size_by_day` | Gold fact and order dimension; day/hour output built before daily output |
| 23 | Reorder Rates | `analytics_reorder_rates` | Gold fact and product dimension |
| 24 | Product Pairs | `analytics_product_pairs` | Gold fact and product dimension |
| 25 | KPIs and Validation | `analytics_kpis`, then a seven-row validation report | Gold fact and all six outputs from Tasks 21–24 |

**Job DAG:** Tasks 21–24 are logically independent of one another and
each sets its own catalog/schema context. Task 25 must wait for all four.

### Design for cost and scale

- `CREATE OR REPLACE TABLE` materializes each answer as a reusable
  snapshot. Dashboard refreshes can read these smaller outputs rather
  than repeatedly performing the large Gold aggregations.
- Task 22 derives its daily summary from its day/hour output to avoid
  repeating the large fact-to-order join.
- Task 23 uses a 500-line eligibility threshold; Task 24 limits candidate
  products to the top 200. Product rankings and pair results are stored
  as top-50 outputs.
- Full refreshes still scan Gold, and pair generation can be expensive.
  There is no guaranteed speedup ratio or guarantee of constant dashboard
  performance as data grows.
- Combining two builds into one task organizes one business question;
  it does not automatically combine their computation into one scan.

### Metric definitions

A purchase is one product line, not physical quantity.
`add_to_cart_order` is basket position and is not used as quantity.

Distinct order counts are not generally additive across products or
departments because the same order can contain several of them. The
KPI query calculates its own global distinct counts from Gold.

## Part 1: Build

Tasks 21–24 create six tables answering the four business questions.
Two outputs for one question remain separate when they have different
grains. The preview SELECTs show results; they are not validation checks.

The original SQL cells and numbering are retained unchanged.

### Question 1: Which products and departments are purchased most frequently?
#### Task 21 — Analytics Product and Department Frequency

Produces two views of purchase frequency:

- `analytics_top_departments`: one row per department, using
  `department_id` and `department_name`.
- `analytics_top_products`: up to 50 product rows ranked by
  `order_line_count`, grouped by `product_id` and descriptive attributes.

Both report product-line purchases and distinct orders. The product
dimension provides display names and hierarchy attributes.

**Depends on:** Gold fact and product dimension.  
**Dashboard use:** Department bar chart and top-products bar chart.

The top-products table is truncated. Its counts do not represent the
entire catalog, and its total should not be expected to equal the
all-data KPI. Apply chart sorting explicitly when consuming stored tables.

In [0]:
-- Owner: Maeve
-- Name: 21 - Analytics Product and Department Frequency
-- Purpose: Identify the products and departments purchased most frequently.
-- Grain: Two outputs: one row per department and one row per product, limited to the top 50 products.

USE CATALOG workspace;
USE SCHEMA instacart_analytics;

-- Department-level answer
CREATE OR REPLACE TABLE analytics_top_departments AS
SELECT
  p.department_id,
  p.department_name,
  COUNT(*) AS order_line_count,
  COUNT(DISTINCT f.order_id) AS distinct_orders
FROM workspace.instacart_gold.gold_fact_order_product f
INNER JOIN workspace.instacart_gold.gold_dim_product p
  ON f.product_id = p.product_id
GROUP BY
  p.department_id,
  p.department_name;

-- Product-level answer
CREATE OR REPLACE TABLE analytics_top_products AS
SELECT
  p.product_id,
  p.product_name,
  p.department_name,
  p.aisle_name,
  COUNT(*) AS order_line_count,
  COUNT(DISTINCT f.order_id) AS distinct_orders
FROM workspace.instacart_gold.gold_fact_order_product f
INNER JOIN workspace.instacart_gold.gold_dim_product p
  ON f.product_id = p.product_id
GROUP BY
  p.product_id,
  p.product_name,
  p.department_name,
  p.aisle_name
ORDER BY order_line_count DESC
LIMIT 50;

-- Preview the department-level answer
SELECT *
FROM analytics_top_departments
ORDER BY order_line_count DESC;

-- Preview the product-level answer
SELECT *
FROM analytics_top_products
ORDER BY order_line_count DESC;

### Question 2: How does customer purchasing behavior change by day of week and hour of day?
#### Task 22 — Analytics Customer Behavior by Day and Hour

Produces two related outputs:

- `analytics_day_hour_patterns`: one row per day code/name and hour,
  containing product-line purchases and distinct orders.
- `analytics_basket_size_by_day`: one row per day code/name, containing
  daily totals and `avg_items_per_order`.

Day and hour come from `gold_dim_order`, not from the narrow fact.
The daily output reuses the day/hour summary and calculates:

```text
average product lines per order = daily product lines / daily orders
```

Summing the hourly distinct-order counts is valid here because each order
belongs to one day/hour bucket, assuming the order dimension has unique,
consistent order records. The query divides summed counts; it does not
average the hourly averages.

**Depends on:** Gold fact and order dimension; daily output depends on
the day/hour output created earlier in the same task.  
**Dashboard use:** Heatmap and average basket-size chart.

Label `avg_items_per_order` as **Average Product Lines per Order**.
Retain `order_dow` for sorting and confirm Gold's day-name mapping before
presenting those names as established weekdays.

In [0]:
-- Owner: Maeve
-- Name: 22 - Analytics Customer Behavior by Day and Hour
-- Purpose: Analyze purchasing patterns by day and hour and calculate average product lines per order by day.
-- Grain: Two outputs: one row per day/hour combination and one row per day.

USE CATALOG workspace;
USE SCHEMA instacart_analytics;

-- Day-and-hour purchasing patterns
CREATE OR REPLACE TABLE analytics_day_hour_patterns AS
SELECT
  o.order_dow,
  o.order_day_name,
  o.order_hour_of_day,
  COUNT(*) AS order_line_count,
  COUNT(DISTINCT f.order_id) AS distinct_orders
FROM workspace.instacart_gold.gold_fact_order_product f
INNER JOIN workspace.instacart_gold.gold_dim_order o
  ON f.order_id = o.order_id
GROUP BY
  o.order_dow,
  o.order_day_name,
  o.order_hour_of_day;

-- Daily basket size calculated from the day/hour summary
CREATE OR REPLACE TABLE analytics_basket_size_by_day AS
SELECT
  order_dow,
  order_day_name,
  SUM(order_line_count) AS order_line_count,
  SUM(distinct_orders) AS distinct_orders,
  TRY_DIVIDE(
    SUM(order_line_count),
    SUM(distinct_orders)
  ) AS avg_items_per_order
FROM analytics_day_hour_patterns
GROUP BY
  order_dow,
  order_day_name;

-- Preview the day/hour patterns in chronological order
SELECT *
FROM analytics_day_hour_patterns
ORDER BY
  order_dow,
  order_hour_of_day;

-- Preview the daily basket sizes
SELECT *
FROM analytics_basket_size_by_day
ORDER BY
  order_dow;

### Question 3: Which products have the highest reorder behavior?
#### Task 23 — Analytics Reorder Rates

Calculates order-line count, reordered-line count, and reorder rate.
The rate is the share of lines where the boolean `reordered` flag is true.

Only groups with at least 500 order lines qualify; the output retains up
to 50 groups ranked by reorder rate, rounded to four decimal places.
The threshold reduces emphasis on tiny samples but does not by itself
establish statistical confidence.

**Depends on:** Gold fact and product dimension; active Analytics schema.  
**Dashboard use:** Reorder-rate ranking with purchase volume displayed.

**Current grouping note:** Although the SQL header describes one row per
product, the query groups by `product_name` and `department_name`, not
`product_id`. Different product IDs with the same names could be merged.
This behavior is documented, not changed.

In [0]:
%sql
-- Owner: Maeve
-- Name: 23 - Analytics Reorder Rates
-- Purpose: Rank products by reorder rate, restricted to products with enough volume for the rate to be meaningful.
-- Grain: One row per product, minimum 500 order-lines, top 50 by reorder rate.

CREATE OR REPLACE TABLE analytics_reorder_rates AS
SELECT
    p.product_name,
    p.department_name,
    COUNT(*) AS total_order_lines,
    SUM(CASE WHEN f.reordered THEN 1 ELSE 0 END) AS reorder_count,
    ROUND(AVG(CASE WHEN f.reordered THEN 1.0 ELSE 0.0 END), 4) AS reorder_rate
FROM workspace.instacart_gold.gold_fact_order_product f
JOIN workspace.instacart_gold.gold_dim_product p ON f.product_id = p.product_id
GROUP BY p.product_name, p.department_name
HAVING COUNT(*) >= 500
ORDER BY reorder_rate DESC
LIMIT 50;

SELECT * FROM analytics_reorder_rates;

### Question 4 (team's question): Which products are most often bought together?
#### Task 24 — Analytics Product Pairs

Selects the 200 most-purchased product IDs, filters fact rows to those
products, and self-joins the filtered lines on `order_id`.
The condition `f1.product_id < f2.product_id` excludes self-pairs and
keeps A–B without also counting B–A.

The output retains up to 50 pairs ranked by `times_bought_together`.
With unique order/product lines, that count represents orders containing
both products.

**Depends on:** Gold fact and product dimension.  
**Dashboard use:** A table or bar chart of frequently co-purchased pairs.

**Scope:** Results cover pairs among the top 200 products, not every
possible product pair. The restriction reduces the candidate product set
but does not guarantee that every useful pairing is included or that
runtime improves by a fixed factor.

**Current grouping note:** The join uses product IDs, but the final
aggregation groups only by the two product names. Different ID pairs
can be merged if names are shared. The current output stores names,
not pair IDs.

Co-purchase frequency is not a measure of causal influence or association
strength such as lift. This task answers the frequency question only.

In [0]:
-- Owner: Maeve
-- Name: 24 - Analytics Product Pairs
-- Purpose: Find product pairs most frequently purchased in the same order, restricted to the
--          top 200 products by volume so the self-join stays tractable at fact-table scale.
-- Grain: One row per unordered product pair (product_a, product_b), top 50 by co-purchase count.

USE CATALOG workspace;
USE SCHEMA instacart_analytics;

CREATE OR REPLACE TABLE analytics_product_pairs AS
WITH top_products AS (
    SELECT product_id
    FROM workspace.instacart_gold.gold_fact_order_product
    GROUP BY product_id
    ORDER BY COUNT(*) DESC
    LIMIT 200
),
filtered_fact AS (
    SELECT f.order_id, f.product_id
    FROM workspace.instacart_gold.gold_fact_order_product f
    JOIN top_products t ON f.product_id = t.product_id
)
SELECT
    p1.product_name AS product_a,
    p2.product_name AS product_b,
    COUNT(*) AS times_bought_together
FROM filtered_fact f1
JOIN filtered_fact f2
    ON f1.order_id = f2.order_id
    AND f1.product_id < f2.product_id
JOIN workspace.instacart_gold.gold_dim_product p1 ON f1.product_id = p1.product_id
JOIN workspace.instacart_gold.gold_dim_product p2 ON f2.product_id = p2.product_id
GROUP BY p1.product_name, p2.product_name
ORDER BY times_bought_together DESC
LIMIT 50;

SELECT * FROM analytics_product_pairs;

## Part 2: Dashboard KPIs and Validation
### Task 25 — Analytics KPIs and Validation

First, this task builds `analytics_kpis` directly from the Gold fact.
It contains one row with:

| KPI | Meaning |
|---|---|
| `total_orders` | Distinct orders represented in the fact |
| `total_order_lines` | Product-line purchase count |
| `total_products` | Distinct products actually purchased, not total product-dimension rows |
| `overall_reorder_rate` | Share of product lines marked reordered, rounded to four decimals |

These are all-data KPIs. They are not obtained by summing distinct counts
or averaging rates from the top-product or department tables.

Then, the query checks all seven Analytics tables:

| Output | Checks performed |
|---|---|
| `analytics_top_departments` | Has rows |
| `analytics_top_products` | Has rows |
| `analytics_day_hour_patterns` | Has rows |
| `analytics_basket_size_by_day` | Has rows |
| `analytics_reorder_rates` | Has rows; rate is nonnull and between 0 and 1 |
| `analytics_product_pairs` | Has rows |
| `analytics_kpis` | Exactly one row; positive counts; valid overall rate; distinct orders/products do not exceed product lines |

**Depends on:** All six business-question outputs and the Gold fact.  
**Expected result:** Seven rows with `status = 'PASS'`.

`invalid_values = 0` is a constant for the five outputs that receive only
a nonempty-table check. It does not mean their measures, keys, or null
values were independently validated.

### What this validation does not prove

These are lightweight sanity checks, not complete reconciliation with
Gold. They do not check every output key, recompute every measure, verify
all limits or thresholds, or test freshness. A `REVIEW` row flags a
problem but does not automatically fail a Job because there is no
`assert_true`.

For a small test dataset, a qualified ranking or product-pair output
could legitimately be empty; the current rule will still return
`REVIEW`. Missing tables cause a SQL error rather than a REVIEW row.

In [0]:
-- Owner: Maeve
-- Name: 25 - Analytics KPIs and Validation
-- Purpose: Build dashboard KPIs and perform sanity checks across all seven analytics tables.
-- Grain: One KPI summary row and one validation summary row per analytics table.

USE CATALOG workspace;
USE SCHEMA instacart_analytics;

-- Build the dashboard KPI summary
CREATE OR REPLACE TABLE analytics_kpis AS
SELECT
  COUNT(DISTINCT order_id) AS total_orders,
  COUNT(*) AS total_order_lines,
  COUNT(DISTINCT product_id) AS total_products,
  ROUND(
    AVG(CASE WHEN reordered = TRUE THEN 1.0 ELSE 0.0 END),
    4
  ) AS overall_reorder_rate
FROM workspace.instacart_gold.gold_fact_order_product;

-- Preview the KPI summary
SELECT *
FROM analytics_kpis;

-- Validate all seven analytics tables
WITH validation AS (
  SELECT
    'analytics_top_departments' AS table_name,
    COUNT(*) AS row_count,
    0 AS invalid_values
  FROM analytics_top_departments

  UNION ALL

  SELECT
    'analytics_top_products',
    COUNT(*),
    0
  FROM analytics_top_products

  UNION ALL

  SELECT
    'analytics_day_hour_patterns',
    COUNT(*),
    0
  FROM analytics_day_hour_patterns

  UNION ALL

  SELECT
    'analytics_basket_size_by_day',
    COUNT(*),
    0
  FROM analytics_basket_size_by_day

  UNION ALL

  SELECT
    'analytics_reorder_rates',
    COUNT(*),
    COUNT_IF(
      reorder_rate IS NULL
      OR reorder_rate < 0
      OR reorder_rate > 1
    )
  FROM analytics_reorder_rates

  UNION ALL

  SELECT
    'analytics_product_pairs',
    COUNT(*),
    0
  FROM analytics_product_pairs

  UNION ALL

  SELECT
    'analytics_kpis',
    COUNT(*),
    COUNT_IF(
      overall_reorder_rate IS NULL
      OR overall_reorder_rate < 0
      OR overall_reorder_rate > 1
      OR total_orders <= 0
      OR total_order_lines <= 0
      OR total_products <= 0
      OR total_orders > total_order_lines
      OR total_products > total_order_lines
    )
  FROM analytics_kpis
)
SELECT
  table_name,
  row_count,
  invalid_values,
  CASE
    WHEN row_count > 0
      AND invalid_values = 0
      AND (
        table_name <> 'analytics_kpis'
        OR row_count = 1
      )
    THEN 'PASS'
    ELSE 'REVIEW'
  END AS status
FROM validation
ORDER BY table_name;

## Analytics Layer: Summary

| Task | Output table | Answers / purpose | Grain and scope |
|---|---|---|---|
| 21 | `analytics_top_departments` | Q1 — departments | One row per department |
| 21 | `analytics_top_products` | Q1 — products | One row per product ID, top 50 |
| 22 | `analytics_day_hour_patterns` | Q2 — when customers shop | One row per day code/name and hour |
| 22 | `analytics_basket_size_by_day` | Q2 — basket-size companion | One row per day code/name |
| 23 | `analytics_reorder_rates` | Q3 — repeat purchasing | One row per product-name/department group; at least 500 lines; top 50 |
| 24 | `analytics_product_pairs` | Q4 — co-purchases | One row per named pair among top-200 product IDs; top 50 |
| 25 | `analytics_kpis` | All-data dashboard counters | One summary row |

**Total:** Five SQL tasks, four business questions, seven output tables,
and seven validation summary rows.

**Expected result:** All seven validation rows return `PASS` for the
intended full dataset. No successful run is claimed by this documentation.

**Next stage:** Dashboards read `workspace.instacart_analytics`.
Materialization reduces repeated aggregation work, but the outputs remain
snapshots and must be refreshed after Gold changes.

### Dashboard interpretation

- All-data KPIs are independent of top-50 ranking tables.
- A top-50 output cannot provide a new global top 50 for a different
  filter context without recomputing that ranking.
- Only connect filters to outputs that contain the required fields.
  These all-data KPI results will not automatically recalculate for
  product or department selections.
- Use numeric day/hour codes for chronological sorting and display
  average basket size as product lines, not quantity.

### Remaining SQL considerations

Tasks 23 and 24 still aggregate by names rather than product IDs.
Their headers describe intended product grains; the actual grouping
behavior and its risks are explained above. That grouping behavior was
not changed here.

*All five original SQL cells, their order, metadata, and outputs were
preserved. Markdown documentation and the copied notebook's display name
were updated. The original notebook was not edited, and no SQL was executed.*